WaveNet

In [2]:
import json
import numpy as np
import soundfile as sf

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

from wavenet import WaveNetModel, mu_law_decode

from IPython.display import Audio


Instructions for updating:
non-resource variables are not supported in the long term


In [3]:
SAMPLES = 16000
TEMPERATURE = 1.0
WAVENET_PARAMS = './wavenet_params_default.json'
WAV_OUT = "generated.wav"

In [7]:
def generate_audio(model_dir, wavenet_params):
    net = WaveNetModel(
        batch_size=1,
        dilations=wavenet_params['dilations'],
        filter_width=wavenet_params['filter_width'],
        residual_channels=wavenet_params['residual_channels'],
        dilation_channels=wavenet_params['dilation_channels'],
        quantization_channels=wavenet_params['quantization_channels'],
        skip_channels=wavenet_params['skip_channels'],
        use_biases=wavenet_params['use_biases'],
        scalar_input=wavenet_params['scalar_input'],
        initial_filter_width=wavenet_params['initial_filter_width'],
        global_condition_channels=None,
        global_condition_cardinality=None
    )

    samples = tf.placeholder(tf.int32)

    next_sample = net.predict_proba(samples)

    decode = mu_law_decode(
        samples,
        wavenet_params["quantization_channels"]
    )

    sess = tf.Session()

    variables_to_restore = {
        var.name[:-2]: var for var in tf.global_variables()
        if not ('state_buffer' in var.name or 'pointer' in var.name)
    }

    saver = tf.train.Saver(variables_to_restore)
    saver.restore(sess, model_dir)

    print("Model załadowany")

    quantization_channels = wavenet_params["quantization_channels"]

    waveform = (
        [quantization_channels // 2]
        * (net.receptive_field - 1)
    )

    waveform.append(
        np.random.randint(quantization_channels)
    )

    for step in range(SAMPLES):
        if len(waveform) > net.receptive_field:
            window = waveform[-net.receptive_field:]
        else:
            window = waveform
        outputs = [next_sample]

        prediction = sess.run(outputs, feed_dict={samples: window})[0]

        scaled_prediction = np.log(
            np.maximum(prediction, 1e-12)
        ) / TEMPERATURE
        scaled_prediction = (scaled_prediction -
                                np.logaddexp.reduce(scaled_prediction))
        scaled_prediction = np.exp(scaled_prediction)
        if TEMPERATURE == 1.0:
                np.testing.assert_allclose(
                        prediction, scaled_prediction, atol=1e-5,
                        err_msg='Prediction scaling at temperature=1.0 '
                                'is not working as intended.')

        sample = np.random.choice(
            np.arange(quantization_channels),
            p=scaled_prediction
        )

        waveform.append(sample)

    audio = sess.run(
        decode,
        feed_dict={samples: waveform}
    )

    return audio
    

Defaultowe parametry (10k kroków)

In [5]:
with open(WAVENET_PARAMS, 'r') as config_file:
    wavenet_params = json.load(config_file)

In [5]:
model_dir = './logdir/train/2026-05-19T19-24-02/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Deprecated in favor of operator or tf.math.divide.
Instructions for updating:
Use `tf.cast` instead.

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-19T19-24-02/model.ckpt-9999
Model załadowany


In [6]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Uproszczone parametry 

In [7]:
with open('./wavenet_params.json', 'r') as config_file_light:
    wavenet_params_light = json.load(config_file_light)

In [12]:
model_dir = './logdir/train/2026-05-19T14-46-50/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params_light)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-19T14-46-50/model.ckpt-9999
Model załadowany


In [13]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Spektralna funkcja straty

In [10]:
model_dir = './logdir/train/2026-05-20T16-01-29/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-20T16-01-29/model.ckpt-9999
Model załadowany


In [11]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Mu-law (mu=63, quantization_channels=64)

In [35]:
wavenet_params["quantization_channels"] = 64

In [36]:
model_dir = './logdir/train/2026-05-21T16-19-16/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-21T16-19-16/model.ckpt-9999
Model załadowany


In [37]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Mu-law (mu=511, quantization_channels=512)

In [40]:
wavenet_params["quantization_channels"] = 512

In [41]:
model_dir = './logdir/train/2026-05-22T14-10-50/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-22T14-10-50/model.ckpt-9999
Model załadowany


In [42]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

In [9]:
tf.reset_default_graph()

<p>
<p>

<p>
<p>

## Zamiana Gated Activation (Sebastian Pietrykowski)

### Omówienie metody

W architekturze WaveNet [Aaron, van den Oord i in.. *WAVENET: A GENERATIVE MODEL FOR RAW AUDIO*. 2016] Gated Activation Units służą do kontrolowania przepływu informacji w warstwach konwolucyjnych. Zamiast jedynie przetwarzać sygnał w sposób liniowy lub statycznie nieliniowy, mechanizm ten pozwala warstwom „decydować”, które fragmenty informacji są w danym momencie istotne dla dalszych obliczeń.

Dzięki temu model może lepiej dopasowywać się do zmiennej struktury sygnału audio, gdzie znaczenie poszczególnych próbek zależy od kontekstu czasowego.

Rozwiązanie to zostało zaczerpnięte z modelu PixelCNN, w którym podobny mechanizm bramkowania wspierał modelowanie zależności sekwencyjnych.

W artykule [Aaron, van den Oord i in.. *WAVENET: A GENERATIVE MODEL FOR RAW AUDIO*. 2016] zauważono, że do zadania modelowania audio funkcja aktywacji `tanh_gated` okazała się znacząco lepsza niż Rectified linear unit (`ReLU`) dzięki swojej nielinowej naturze. W ramach eksperymentów warto zatem rozważyć głównie funkcje nieliniowe.

<img src="./Sprawozdanie/images/visualization_of_a_stack_of_dilated_casual_convolutional_layers.png" width="500">


### Bibliografia
- Aaron, van den Oord i in.. *WAVENET: A GENERATIVE MODEL FOR RAW AUDIO*. 2016 (https://arxiv.org/pdf/1609.03499)
- van den Oord, Aaaron, Kalchbrenner, Nal, Vinyals, Oriol, Espeholt, Lasse, Graves, Alex, Kavukcuoglu, Koray. *Conditional image generation with PixelCNN decoders*. 2016. (http://arxiv.org/abs/1606.05328)
- Nair, Vinod, Hinton, Geoffrey E. *Rectified linear units improve restricted Boltzmann machines*. W ICML, s. 807–814, 2010.
- Yann, Dauphin i in.. *Language Modeling with Gated Convolutional Networks*. 2016. (https://arxiv.org/abs/1612.08083)
- Noam, Shazeer. *GLU Variants Improve Transformer*. 2020. (https://arxiv.org/abs/2002.05202)

### Wspólna kongiguracja
- Liczba próbek: 16k
- Liczba kroków: 20k

<p>


### **Wariant bazowy - Gated Tanh Unit (GTU)**

##### Omówienie wariantu

Wariant bazowy wykorzystuje oryginalną funkcję aktywacji z architektury WaveNet, czyli bramkowaną jednostkę aktywacji (ang. Gated Tanh Unit – GTU). Konstrukcja ta opiera się na matematycznym podziale na dwie niezależne gałęzie: filtr reprezentowany przez funkcję $\tanh$ (odpowiedzialny za ekstrakcję cech i nieliniowe mapowanie sygnału w zakresie $[-1, 1]$) oraz bramkę opartą na sigmoidzie $\sigma$ (kontrolującą przepływ informacji w przedziale $[0, 1]$). Dzięki takiemu podejściu model zyskuje wysoką ekspresyjność nieliniową, zdolność do selektywnego zapamiętywania kontekstu oraz stabilny przepływ gradientu, co skutecznie zapobiega jego zanikaniu w głębokich strukturach splotowych.

##### wavenet_params.json - konfiguracja bazowa

```json
"gated_activation": "tanh_gated"
```

##### Wzór

$z = \tanh(W_f * x) \odot \sigma(W_g * x)$

<img src="./Sprawozdanie/images/wykres_tanh_gated.png" width="500">

##### Najlepszy wynik

In [ ]:
Audio(filename='./Sprawozdanie/audio/default_20k_steps_audio_v3_fast.wav')

##### Ocena

Nie jest to biału szum, ale szum jest dalej wyraźnie wyczuwalny. Przypomina charakterstyką mowę, ale w zatłoczonym miejscu i bardzo zniekształconą.

<p>

### **Wariant Gated Linear Unit (GLU)**

##### Omówienie wariantu

W tym wariancie klasyczna funkcja $\tanh$ w gałęzi głównej została zastąpiona czysto liniową transformacją, co odpowiada klasycznej architekturze bramkowanej jednostki liniowej (ang. Gated Linear Unit – GLU). Brak nieliniowego ograniczenia wartości w filtrującej części układu pozwala na niemal nieograniczoną przepustowość sygnału dla dodatnich wartości wejściowych oraz linearyzację przepływu gradientu. Choć uproszczenie to redukuje złożoność obliczeniową i ułatwia trening głębokich sieci eliminując problem nasycania się aktywacji, jednocześnie osłabia ono nieliniową selekcję cech w warstwach splotowych w porównaniu do klasycznego podejścia opartego na $\tanh$.

##### Zmiana w wavenet_params.json względem konfiguracji bazowej
```json
"gated_activation": "glu"
```

##### Wzór

$z = (W_f * x) \odot \sigma(W_g * x)$

<img src="./Sprawozdanie/images/wykres_glu.png" width="500">

##### Najlepszy wynik

In [38]:
Audio(filename='./Sprawozdanie/audio/gated_activation_glu_steps_20k_no_5.wav')

##### Ocena

Całkowiecie biały szum. Wynik do odrzucenia.

<p>

### **Wariant Swish-Gated Linear Unit (SwiGLU)**

##### Omówienie wariantu

W tym wariancie w gałęzi filtrującej zastosowano nowoczesną funkcję aktywacji Swish ($x \cdot \sigma(x)$), co prowadzi do uzyskania struktury typu SwiGLU (Swish-Gated Linear Unit). Konstrukcja ta łączy najlepsze cechy dwóch poprzednich podejść: gładką, nieliniową charakterystykę znaną z $\tanh$ oraz brak twardego ograniczenia wartości dla dużych aktywacji dodatnich, co cechuje wariant GLU. SwiGLU charakteryzuje się nieznaczną asymetrią oraz unikalną, niemonotoniczną „dolinką” dla małych wartości ujemnych. Taka geometria pozwala na płynniejszy przepływ informacji i stabilizację procesu uczenia, dzięki czemu architektury typu Swish-gated wyparły klasyczne rozwiązania w wielu najnowocześniejszych modelach głębokiego uczenia.


##### Zmiana w wavenet_params.json względem konfiguracji bazowej
```json
"gated_activation": "swish_gated"
```

##### Wzór

$z = \text{swish}(W_f * x) \odot \sigma(W_g * x), \quad \text{gdzie} \quad \text{swish}(x) = x \cdot \sigma(x)$

<img src="./Sprawozdanie/images/wykres_swish_gated.png" width="500">

##### Najlepszy wynik

In [ ]:
Audio(filename='./Sprawozdanie/audio/gated_activation_swish_gated_steps_20k_no_5.wav')

##### Ocena

Nie jest to całkowicie biały szum. Dźwięki generowane przez ten model przypominają ciężki sprzęt.

### **Wnioski z eksperymentu**

Najlepsze rezulataty uzyskano używając domyślnej funkcji gated activation: `tanh-gated`. Uzyskana próbka ma cechy charakterystyczne dla mowy. W eksperymentach mechanizm ten okazał się skuteczniejszy dla danych audio niż klasyczne aktywacje typu ReLU, ponieważ lepiej modeluje zależności nieliniowe i strukturę sygnału.

Badano także funkcje `Gated Activation Unit (GLU)`, dla której uzyskano wyłącznie szum, oraz `swish-gated`, dla której otrzymano dźwięk przypominający cięzki sprzęt zamiast mowy.



<p>
<p>

## Zmiana kroku dylatacji w warstwach (Sebastian Pietrykowski)

### Omówienie metody

Architektura WaveNet wykorzystuje 'dilated causal convolutions' (dylatowane sploty przyczynowe) do modelowania sygnału audio. Mechanizm ten pozwala sieci analizować próbki z coraz większego zakresu czasowego bez konieczności znaczącego zwiększania liczby parametrów modelu.

Splot przyczynowy ('causal convolution') zapewnia, że predykcja próbki w chwili `t` zależy wyłącznie od wcześniejszych próbek `(t-1, t-2, ...)`, a nie od przyszłych wartości sygnału. Dzięki temu model może generować dźwięk sekwencyjnie i zachowuje poprawny porządek czasowy danych.

Dylatacja (dilation) określa krok pomiędzy próbkami analizowanymi przez filtr splotowy:
- `dilation = 1` - oznacza analizę każdej kolejnej próbki,
- `dilation = 2` - oznacza analizę co drugiej próbki,
- `dilation = 4` - co czwartej próbki itd.

Rosnąca dylatacja zwiększa 'receptive field' modelu, czyli zakres próbek wejściowych wpływających na pojedynczą próbkę wyjściową. W WaveNet standardowo dylatacja jest podwajana dla każdej warstwy rosnąco aż do osiągnięcia limitu, a następnie powtarzana. Przykład: `1, 2, 4, ..., 512, 1, 2, 3, 4, ..., 512, 1, 2, 3, 4, ..., 512`. 


Można wymienić dwa powody dla użycia takiego podejścia. Po pierwsze, wykładnicze zwiększanie współczynnika dylatacji prowadzi do wykładniczego wzrostu pola recepcyjnego wraz z głębokością sieci. Na przykład każdy blok o dylatacjach `1, 2, 4, ..., 512` ma pole recepcyjne o rozmiarze `1024` i może być traktowany jako bardziej wydajny i bardziej selektywny (nieliniowy) odpowiednik splotu `1×1024`. Po drugie, układanie takich bloków jeden na drugim dodatkowo zwiększa pojemność modelu oraz rozmiar pola recepcyjnego.


### Interpretacja parametru `dilations`

W konfiguracji WaveNet parametr:
```json
"dilations": [1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
```
opisuje układ warstw dylatowanych w sieci.

Interpretacja wymiarów:

- każda liczba oznacza wartość dylatacji dla pojedynczej warstwy,
- pojedynczy ciąg: `1, 2, 4, 8, 16, 32, 64, 128, 256, 512` tworzy jeden blok dylatacyjny,
- liczba kolumn odpowiada liczbie warstw w pojedynczym bloku,
- liczba rzędów odpowiada liczbie powtórzeń całego bloku.

### Opis przeprowadzonych eksperymentów

W przeprowadzonych eksperymentach zmieniano:
- maksymalną wartość dylatacji,
- liczbę bloków,
- szerokość pojedynczego cyklu dylatacji.

Celem było sprawdzenie, jak zmiana pola recepcyjnego wpływa na jakość generowanego dźwięku.

### Bibliografia
- Aaron, van den Oord i in.. *WAVENET: A GENERATIVE MODEL FOR RAW AUDIO*. 2016 (https://arxiv.org/pdf/1609.03499)
- Yu, Fisher i Koltun, Vladlen. *Multi-scale context aggregation by dilated convolutions*. W ICLR, 2016. (http://arxiv.org/abs/1511.07122)

### Wspólna kongiguracja
- Liczba próbek: 16k
- Liczba kroków: 20k

<p>

### **Wariant bazowy - standardowy zakres dylatacji (1–512)**

##### wavenet_params.json - konfiguracja bazowa
```json
   "dilations": [1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
                 1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
                 1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
                 1, 2, 4, 8, 16, 32, 64, 128, 256, 512,
                 1, 2, 4, 8, 16, 32, 64, 128, 256, 512], # 5 rzędów, 10 kolumn
```

##### Najlepszy wynik

In [36]:
Audio(filename='./Sprawozdanie/audio/default_20k_steps_audio_v3_fast.wav')

<p>

### **Wariant 1 - większa maksymalna dylatacja, mniej bloków**

##### Omówienie wariantu

W tym wariancie zwiększono maksymalną wartość dylatacji do 1024 z 512, jednocześnie zmniejszając liczbę powtórzeń całego bloku z 5 do 3. Model otrzymał większe pole recepcyjne w pojedynczym cyklu, ale mniejszą całkowitą głębokość sieci.

##### Zmiana w wavenet_params.json względem konfiguracji bazowej
```json
"dilations": [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024] # 3 rzędy, 11 kolumn
```
Mniej bloków (3 zamiast 5), ale każdy szerszy. 


##### Najlepszy wynik

In [35]:
Audio(filename='./Sprawozdanie/audio/dilations_1_steps_20k_no_3.wav')

##### Ocena

Całkowiecie biały szum. Wynik do odrzucenia.

<p>

### **Wariant 2 - większa maksymalna dylatacja, zachowana liczba bloków**

##### Omówienie wariantu

Wariant rozszerza maksymalną dylatację z 512 do 1024, zachowując jednocześnie pięć pełnych cykli dylatacji - tak jak w konfiguracji bazowej. Powoduje to znaczące zwiększenie pola recepcyjnego całego modelu oraz wzrost liczby warstw analizujących odległe zależności czasowe.

##### Zmiana w wavenet_params.json względem konfiguracji bazowej
```json
"dilations": [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024,
              1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024] # 5 rzędów, 11 kolumn
```

##### Najlepszy wynik

In [34]:
Audio(filename='./Sprawozdanie/audio/dilations_2_steps_20k_no_2.wav')

##### Ocena

Nie ma ciągłego szumu, chwilami jest cisza. Dźwięk przypomina zgniatanie.

<p>

### **Wariant 3 - mniejsza maksymalna dylatacja, więcej bloków**

##### Omówienie wariantu

W tym wariancie zmniejszono maksymalną wartość dylatacji z 512 do 256, ale zwiększono liczbę powtórzeń cyklu do sześciu. Dzięki temu model częściej analizuje lokalne zależności w sygnale, jednocześnie zachowując dużą głębokość sieci.

##### Zmiana w wavenet_params.json względem konfiguracji bazowej
```json
"dilations": [1, 2, 4, 8, 16, 32, 64, 128, 256,
              1, 2, 4, 8, 16, 32, 64, 128, 256,
              1, 2, 4, 8, 16, 32, 64, 128, 256,
              1, 2, 4, 8, 16, 32, 64, 128, 256,
              1, 2, 4, 8, 16, 32, 64, 128, 256,
              1, 2, 4, 8, 16, 32, 64, 128, 256] # 6 rzędów, 9 kolumn
```

##### Najlepszy wynik

In [33]:
Audio(filename='./Sprawozdanie/audio/dilations_3_steps_20k_no_2.wav')

##### Ocena

Nie ma monotonnego szumu. Dźwięki generowane przez ten model przypominają syczenie przez człowieka.

<p>

### **Wnioski z eksperymentu**

Najlepsze rezultaty uzyskano dla konfiguracji bazowej (dylatacja 1–512) oraz wariantu 3 z mniejszą maksymalną dylatacją (256) i większą liczbą bloków. W obu przypadkach model generował sygnał z wyraźną strukturą przypominającą mowę, mimo obecności szumu i zniekształceń.

Warianty z maksymalną dylatacją zwiększoną do 1024, szczególnie przy mniejszej liczbie bloków, prowadziły do całkowitej degradacji jakości sygnału i generowały głównie biały szum. W przypadku utrzymania większej liczby bloków pojawiały się jedynie lokalne fragmenty struktury dźwięku, jednak bez spójnej ciągłości.

Najbardziej stabilne zachowanie modelu obserwowano wtedy, gdy zachowana była równowaga między głębokością sieci a wielkością pojedynczego bloku dylatacji. Zbyt duże pole recepcyjne w jednym cyklu powodowało utratę lokalnych zależności w sygnale, natomiast zbyt mała liczba bloków ograniczała zdolność modelu do uchwycenia długoterminowej struktury audio.


<p>

## Ogólne uwagi


### Karl, Hiner. *Generating Music with WaveNet and SampleRNN* (Sebastian Pietrykowski)

Wpis dokumentuje eksperymenty z bezwarunkowym generowaniem muzyki akustycznej za pomocą dwóch modeli autoregresyjnych: WaveNet (sploty rozszerzone) oraz SampleRNN (sieci rekurencyjne). Autor trenował je na minimalistycznych utworach zespołu Dawn of Midi.

Autor trenował modele WaveNet do osiągnięcia 100k kroków (czyli domyślna wartość w WaveNet). Taki wynik był u nas nieosiągalny ze względu na ograniczenia sprzętowe. W naszym przypadku uczenie jednego modelu trwałoby wtedy ok. 30 h. Tym samym warto omówić wyniki uzyskane przez autora.

Wygenerowane przez autora próbki używając WaveNet były znacznie lepszej jakości niż nasze, które zostały wygenerowane na modelach trenowanych do osiągnięcia 8-20k kroków, co może świadczyć o decydującej roli liczby kroków podczas uczenia.

- WaveNet generował czystszy dźwięk z mniejszym szumem niż SampleRNN. Wynika to z użycia kompandingu $\mu$-law, podczas gdy SampleRNN cierpiał na cyfrowe zniekształcenia (błędy kwantyzacji liniowej).
- Podjęto próbę zwiększenia pola receptywnego w WaveNet poprzez dodanie więcej warstw dylatacji, aby poprawić rozumienie zaszumionego i złożonego zestawu treningowego:
  
  ```json"
  dilations": [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1014, 2048,
               1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048,
               1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048,
               1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048,
               1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048,
               1, 2, 4, 8, 16, 32, 64]
  ```
  Autor uzyskał poprawę. Rezultaty były nadal dość kakofoniczne, ale występowały punkty przejściowe, krótkie pauzy oraz wysokie tony i piski. Dźwięki perkusyjne dalej brzmiały bardziej jak parodia, ale nie był to już biały szum.
- Autor eksperymentował także ze zwiększeniem liczby kroków z 100k do 200k oraz liczby próbek z 16k do 320k i 640k
- SampleRNN zdecydowanie wygrał pod względem kompozycji. Dzięki hierarchicznym warstwom RNN potrafił utrzymać długoterminowy rytm, pauzy i dynamikę, podczas gdy WaveNet szybko gubił wątek.
- Trening SampleRNN był niestabilny (generował trzaski). Autor rozwiązał to, wprowadzając temperaturę próbkowania na poziomie 0.95, co zapobiegło eksplozji amplitudy sygnału.




##### Najlepszy wynik

In [4]:
Audio(filename='./Sprawozdanie/audio/Dom_-_Largemodel_-_200ksteps_-_320000samples_KLICKAUD.mp3')

### Bibliografia
- Karl, Hiner. *Generating Music with WaveNet and SampleRNN*. Wpis na blogu. (https://karlhiner.com/music_generation/wavenet_and_samplernn/)